In [ ]:
!pip -q install torch transformers evaluate datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.3 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset
data = load_dataset('sh0416/ag_news')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/2.08k [00:00<?, ?B/s]

train.jsonl:   0%|          | 0.00/33.7M [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [ ]:
data.column_names

{'train': ['label', 'title', 'description'],
 'test': ['label', 'title', 'description']}

#Label_names

0 = World
1 = Sports
2 = Business
3 = Sci/Tech



In [ ]:
data['train'].shape

(120000, 3)

In [ ]:
data['test'][100]

{'label': 2,
 'title': 'Olympic history for India, UAE',
 'description': 'An Indian army major shot his way to his country #39;s first ever individual Olympic silver medal on Tuesday, while in the same event an member of Dubai #39;s ruling family became the first ever medallist from the United Arab Emirates. '}

In [ ]:
data['train'][10000]

{'label': 1,
 'title': 'A Daily Look at U.S. Iraq Military Deaths (AP)',
 'description': 'AP - As of Wednesday, Aug. 25, 964 U.S. service members have died since the beginning of military operations in Iraq in March 2003, according to the Defense Department. Of those, 722 died as a result of hostile action and 242 died of non-hostile causes.'}

In [ ]:
data = data.map(lambda x:{'label':x['label']-1})

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [ ]:
data['train'][10000]

{'label': 0,
 'title': 'A Daily Look at U.S. Iraq Military Deaths (AP)',
 'description': 'AP - As of Wednesday, Aug. 25, 964 U.S. service members have died since the beginning of military operations in Iraq in March 2003, according to the Defense Department. Of those, 722 died as a result of hostile action and 242 died of non-hostile causes.'}

In [ ]:
data['test'][100]

{'label': 1,
 'title': 'Olympic history for India, UAE',
 'description': 'An Indian army major shot his way to his country #39;s first ever individual Olympic silver medal on Tuesday, while in the same event an member of Dubai #39;s ruling family became the first ever medallist from the United Arab Emirates. '}

In [ ]:
from transformers import AutoTokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    # "bert-base-uncased"
   "roberta-base"
)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [ ]:
demo_text = "Hi this is Omm Tripathi"
tokenizer.tokenize(demo_text)
# bert uses sub word tokenization

['Hi', 'Ġthis', 'Ġis', 'ĠO', 'mm', 'ĠTrip', 'athi']

In [ ]:
tokens = tokenizer(demo_text)
tokens

{'input_ids': [0, 30086, 42, 16, 384, 5471, 13734, 20206, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [ ]:
# tokenizeing the entire dataset
def tokenize(batch):
  return tokenizer(
      batch['title'],
      batch["description"],
      padding=True,# this adds a padding to the text to make it of same length
      truncation=True,# Bert has a input size of 512 is the text exceedes that we have to reduce it
      max_length = 512
  )

In [ ]:
dataset = data.map(tokenize,batched = True)

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['label', 'title', 'description', 'input_ids', 'attention_mask'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['label', 'title', 'description', 'input_ids', 'attention_mask'],
        num_rows: 7600
    })
})

In [ ]:
print(dataset["train"][0].keys())

dict_keys(['label', 'title', 'description', 'input_ids', 'attention_mask'])


In [ ]:
print(len(dataset["train"][0]["input_ids"]))
print(len(dataset["train"][1]["input_ids"]))

318
318


In [ ]:
dataset = dataset.remove_columns(
    ["title", "description"]
)

dataset = dataset.rename_column(
    "label",
    "labels"
)

In [ ]:
print(dataset["train"][0].keys())

dict_keys(['labels', 'input_ids', 'attention_mask'])


# For now we are directly downloading the model with the classification head in a function

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    # "bert-base-uncased",
    "roberta-base",
    num_labels=4
)

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
print(model.device)
# print(torch.cuda.get_device_name(0))

cpu


In [ ]:
import torch
torch.cuda.is_available()
device = torch.device("cuda")

model = model.to(device)

print(model.device)

cuda:0


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs = 3,
    per_device_train_batch_size = 32,
    per_device_eval_batch_size = 32,
    learning_rate = 2e-5,
    eval_strategy ="epoch"
)

In [32]:
# trainer api
from transformers import Trainer
train_subset = dataset["train"].shuffle(seed=42).select(range(75000))# coz collab ka ghee khatam hojayega
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_subset,
    eval_dataset = dataset['test'],
    data_collator=data_collator
)

In [33]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.204640,0.193798
2,0.140959,0.173048
3,0.112021,0.186379


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=7032, training_loss=0.16048530191691662, metrics={'train_runtime': 15639.7078, 'train_samples_per_second': 14.386, 'train_steps_per_second': 0.45, 'total_flos': 3.949581640812288e+16, 'train_loss': 0.16048530191691662, 'epoch': 3.0})

In [35]:
predictions = trainer.predict(
    dataset["test"]
)

In [36]:
import evaluate
import numpy as np

metric = evaluate.load("accuracy")

preds = np.argmax(
    predictions.predictions,
    axis=1
)

metric.compute(
    predictions=preds,
    references=predictions.label_ids
)

{'accuracy': 0.9476315789473684}

In [44]:
text = "Australia spoils Turkiye’s return to World Cup stage, wins 2-0 The ​result puts Australia second in the ⁠group behind United States after ⁠the co-hosts' 4-1 win over Paraguay"

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True
)

inputs = {k: v.to(model.device) for k, v in inputs.items()}

outputs = model(**inputs)

In [45]:
outputs

SequenceClassifierOutput(loss=None, logits=tensor([[ 0.0985,  6.3712, -2.7982, -2.8199]], device='cuda:0',
       grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)

In [48]:
trainer.save_model("./News_Classifier_Model")
tokenizer.save_pretrained("./News_Classifier_Model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./News_Classifier_Model/tokenizer_config.json',
 './News_Classifier_Model/tokenizer.json')

Uploading to HF Hub

In [50]:
from huggingface_hub import login

login("")

model.push_to_hub("News_Classifier_Model")
tokenizer.push_to_hub("News_Classifier_Model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...1sberls/model.safetensors:   2%|2         | 11.7MB /  499MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/MhoOmm/News_Classifier_Model/commit/48c4b9167a63987973db339d452206012582ed50', commit_message='Upload tokenizer', commit_description='', oid='48c4b9167a63987973db339d452206012582ed50', pr_url=None, repo_url=RepoUrl('https://huggingface.co/MhoOmm/News_Classifier_Model', endpoint='https://huggingface.co', repo_type='model', repo_id='MhoOmm/News_Classifier_Model'), pr_revision=None, pr_num=None)